# PI-GNODE Cross-Region Generalization on Colab

Trains PI-GNODE on high-elevation NDWS events, then on low-elevation, then evaluates each model on both regions.  Output: 4-cell metric matrix + heatmap PNG.

**Runtime:** Google Colab free tier (T4 GPU). Total wall-clock ~1.5–2 hours.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Clone the repo and install dependencies

In [ ]:
%cd /content
REPO_URL = 'https://github.com/josephyu12/pignode-wildfire.git'  # change if your fork

!rm -rf /content/pignode-wildfire
!git clone {REPO_URL} /content/pignode-wildfire
%cd /content/pignode-wildfire

import sys, os
sys.path.insert(0, os.path.abspath('src'))

In [ ]:
!pip install -q xarray zarr scikit-learn matplotlib pandas tqdm pyyaml huggingface-hub
!pip install -q torch-geometric torchdiffeq

import torch
print(f'torch={torch.__version__}  cuda={torch.version.cuda}  gpu={torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
import wildfire
from wildfire.data.ndws import compute_norm_stats, compute_region_assignments
print('wildfire package imports OK')

## 2. Download NDWS data

From HuggingFace (~2.4 GB total).  Takes ~5–10 min.

In [ ]:
import os
os.makedirs('data/raw/ndws', exist_ok=True)

from huggingface_hub import snapshot_download
snap = snapshot_download(
    repo_id='TheRootOf3/next-day-wildfire-spread',
    repo_type='dataset',
    local_dir='data/raw/ndws/_hf',
    allow_patterns=['*.zarr.zip', '*.zip'],
)
print('downloaded to', snap)

import zipfile, glob
for split in ('train', 'eval', 'test'):
    matches = glob.glob(f'{snap}/**/{split}*.zip', recursive=True) + \
              glob.glob(f'{snap}/{split}*.zip')
    if not matches:
        raise RuntimeError(f'no zip found for split={split}; check {snap}')
    z = matches[0]
    target = f'data/raw/ndws/{split}.zarr'
    if os.path.isdir(target):
        continue
    print(f'unzipping {z} -> {target}')
    with zipfile.ZipFile(z) as zf:
        zf.extractall(target)

!ls -la data/raw/ndws/

In [ ]:
stats = compute_norm_stats()
print('norm stats computed -> data/processed/norm_stats.npz')

regions = compute_region_assignments()
print('region assignments:')
print('  threshold elevation:', float(regions['threshold']))
for k in regions:
    if k != 'threshold':
        print(f'  {k}: {len(regions[k])} events')

## 3. Train PI-GNODE on each region

Two trainings of 8 epochs each.  ~30–45 min each on T4.

In [ ]:
# High-elevation training
!PYTHONPATH=src python -m wildfire.train --exp pignode_high_elev \
    --model pignode --uniform-edges --hidden 128 --ode-layers 2 \
    --n-eval-steps 1 --t-end 1.0 --epochs 8 --batch-size 16 --lr 3e-4 \
    --eval-batches 30 --region high_elev --device cuda

In [ ]:
# Low-elevation training
!PYTHONPATH=src python -m wildfire.train --exp pignode_low_elev \
    --model pignode --uniform-edges --hidden 128 --ode-layers 2 \
    --n-eval-steps 1 --t-end 1.0 --epochs 8 --batch-size 16 --lr 3e-4 \
    --eval-batches 30 --region low_elev --device cuda

## 4. Cross-evaluate the 2x2 matrix

In [ ]:
for trained in ('high_elev', 'low_elev'):
    for evald in ('high_elev', 'low_elev'):
        ckpt = f'experiments/pignode_{trained}/best.pt'
        print(f'\n=== eval: trained on {trained}, tested on {evald} ===')
        !PYTHONPATH=src python -m wildfire.eval_region --ckpt {ckpt} --region {evald} --split test --device cuda

## 5. Generate the heatmap figure

In [ ]:
!PYTHONPATH=src python -m wildfire.figures
from IPython.display import Image
Image('experiments/_figures/region_split.png')

## 6. Download results back to your computer

Zips just the new region-split outputs (not the raw data) and downloads.

In [ ]:
!zip -r region_split_results.zip \
    experiments/pignode_high_elev/metrics.json \
    experiments/pignode_high_elev/best.pt \
    experiments/pignode_high_elev/eval_test_high_elev.json \
    experiments/pignode_high_elev/eval_test_low_elev.json \
    experiments/pignode_low_elev/metrics.json \
    experiments/pignode_low_elev/best.pt \
    experiments/pignode_low_elev/eval_test_high_elev.json \
    experiments/pignode_low_elev/eval_test_low_elev.json \
    experiments/_figures/region_split.png

from google.colab import files
files.download('region_split_results.zip')